<a href="https://colab.research.google.com/github/akshas-25/NLP-Learnings/blob/main/NLP_SentimentalAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Re-load the data to ensure we start with a clean slate
df=pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])

# 1. Map emotions to numbers (re-initialise unique_emotions from fresh df)
unique_emotions = df['emotion'].unique()
emotion_numbers={}
for i, emo in enumerate(unique_emotions):
  emotion_numbers[emo]=i
df['emotion']=df['emotion'].map(emotion_numbers)

# 2. Convert text to lowercase
df['text']=df['text'].apply(lambda x: x.lower())

# 3. Define and apply function to remove punctuation
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))
df['text']=df['text'].apply(remove_punc)

# 4. Define and apply function to remove numbers
def remove_num(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new
df['text']=df['text'].apply(remove_num)

# 5. Define and apply function to remove emojis (corrected logic)
def remove_emojis(txt):
  new =""
  for i in txt:
    if i.isascii(): # Keep only those characters that have ascii values, effectively removing emojis
      new=new+i
  return new
df['text']=df['text'].apply(remove_emojis)

# 6. Define and apply function to remove stopwords
# Ensure NLTK resources are downloaded (quietly to avoid repeated output)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Ensure punkt_tab is downloaded

stop_words=set(stopwords.words('english'))

def remove(txt):
  # Ensure the text is not empty before tokenizing
  if not txt.strip(): # Check for empty or whitespace-only strings
    return ""
  words=word_tokenize(txt)
  new_words=[]
  for w in words:
    if w not in stop_words:
      new_words.append(w)
  return " ".join(new_words)

df['text']=df['text'].apply(remove)

# Display the cleaned DataFrame head to verify
display(df.head())

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [22]:
from sklearn.model_selection import train_test_split

In [23]:

X_train,X_test,y_train,y_test=train_test_split(df['text'],df['emotion'],test_size=0.20,random_state=42)

In [24]:
X_train

,text
676,refers course though cant help feeling somehow...
12113,im starting feel im suffering fatigue
7077,feel like probably would liked book little bit...
13005,really feel awkward
12123,im feeling little grumpy today lame weather te...
...,...
13418,love leave reader feeling confused slightly de...
5390,feel delicate
860,starting feel little stressed
15795,feel stressed tired worn shape neglected


In [25]:
X_test

,text
8756,ive made week feel beaten
4660,feel strategy worthwhile
6095,feel worthless weak say want find
304,feel clever nov
8241,im moved ive feeling kind gloomy
...,...
15578,feel useful pulpit find ironic often question ...
5746,dried bladders ready day im feeling brave
6395,feel thrilled matter days
7624,woke morning text mr c declaring walking work ...


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [27]:
bow_vectorizer=CountVectorizer()


In [28]:
x_train_bow=bow_vectorizer.fit_transform(X_train)


In [31]:
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116049 stored elements and shape (12800, 13359)>

In [32]:
x_test_bow=bow_vectorizer.transform(X_test)

In [33]:
from sklearn.naive_bayes import MultinomialNB #gaussian is used for normal distribution and hence we have used multinominal

In [34]:
from sklearn.metrics import accuracy_score

In [35]:
nb_model=MultinomialNB() #using naive baiyes model
nb_model.fit(x_train_bow,y_train)

MultinomialNB()

In [36]:
pred_nb=nb_model.predict(x_test_bow)

In [37]:
print(accuracy_score(y_test,pred_nb))

0.7678125


In [38]:
tfidf_vectorizer=TfidfVectorizer() #now doing with the tfidf vectorizer

In [46]:
x_train_tfidf=tfidf_vectorizer.fit_transform(X_train)
x_test_tfidf=tfidf_vectorizer.transform(X_test)

In [47]:
nb2_model=MultinomialNB()
nb2_model.fit(x_train_tfidf,y_train)

MultinomialNB()

In [48]:
y_pred=nb2_model.predict(x_test_tfidf)

In [49]:
print(accuracy_score(y_test,y_pred))

0.6609375


In [50]:
from sklearn.linear_model import LogisticRegression

In [51]:
logistic_model=LogisticRegression(max_iter=1000)
logistic_model.fit(x_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [52]:
log_pred=logistic_model.predict(x_test_tfidf)

In [53]:
print(accuracy_score(y_test,log_pred))

0.8615625
